### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:
- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.

In [ ]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:openai/gpt-oss-20b")
model

### Summarization MiddleWare
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:
- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [18]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Message based summarization
agent=create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages",10), ##triggered after 10 messages history
            keep=("messages",4)
        )
    ]
)

Remember, a message is not necessarily one question. A normal exchange usually creates two messages:

    HumanMessage("What is 2+2?")
    AIMessage("4")

In [19]:
### Run with thread id
config={"configurable":{"thread_id":"test-1"}}

The thread_id identifies the conversation whose state should be loaded and updated.
Think of it like a conversation ID:

thread_id = "test-1"
       │
       ▼
Checkpointed conversation state
       │
       ├── Earlier questions and answers
       ├── Any summary created by middleware
       └── New questions and answers

In [20]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke(
        {"messages": [HumanMessage(content=q)]},
        config=config
    )
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")


Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='09d0b47a-db02-4c7e-8a69-6286677f424f'), AIMessage(content='2\u202f+\u202f2\u202f=\u202f4.', additional_kwargs={'reasoning_content': 'The user asks "What is 2+2?" The answer is 4. It\'s straightforward.'}, response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 78, 'total_tokens': 118, 'completion_time': 0.04517252, 'completion_tokens_details': {'reasoning_tokens': 21}, 'prompt_time': 0.003668587, 'prompt_tokens_details': None, 'queue_time': 0.303100812, 'total_time': 0.048841107}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_d3e146e1a5', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a11eca-77ef-7793-8b0e-9b743b885a92-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 40, 'total_tokens': 118, 'output_token_details': {'reasoning': 21}

### Although you provide only the new user message each time, the checkpointer can restore the previous state for test-1. The middleware can then manage the growing history.

Your experiment

1. Create the agent. Configure the Groq model, checkpointing, and summarization middleware.
2. Send the first question. The agent answers and saves its state under test-1.
3. Send subsequent questions. The same thread ID restores and updates the existing conversation.
4. Reach the summarization threshold. When the trigger condition is met, the middleware summarizes older messages and retains recent history according to its settings.
5. Continue the conversation. The agent can keep working without needing to pass every original message back manually.

## token Size

In [21]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent=create_agent(
    model=model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens",550),
            keep=("tokens",200),
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [ ]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~244 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='38602f00-d197-43c6-9cc6-a2d91f105850'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use the function to search hotels in Paris.', 'tool_calls': [{'id': 'fc_843ba84c-56c9-4f69-9ef3-3d932001c0ba', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 129, 'total_tokens': 166, 'completion_time': 0.03834495, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.006269192, 'prompt_tokens_details': None, 'queue_time': 0.21249162, 'total_time': 0.044614142}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_a8c584dda7', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a11edd-8dfc-7343-a72e-bfb80d719d80-0', tool_calls=[{'name':

### Fraction

In [23]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model=model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~168 tokens (0.1313%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='5cf84f0d-c64b-4442-ab9c-b4cefd455a29'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use function search_hotels with city "Paris".', 'tool_calls': [{'id': 'fc_90a969fd-265a-458a-bfa0-2f044fb0c398', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 120, 'total_tokens': 158, 'completion_time': 0.069731976, 'completion_tokens_details': {'reasoning_tokens': 14}, 'prompt_time': 0.007009832, 'prompt_tokens_details': None, 'queue_time': 0.384985596, 'total_time': 0.076741808}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_99996fee8e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a11f54-35db-75f2-b9ee-1e7293a55ad9-0', tool_calls=[{'n

### Human In the Loop MiddleWare

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:
- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [32]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [33]:
agent=create_agent(
    model=model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,

            }
        )
    ]
)

In [34]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [35]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='c11a907b-9978-4033-a81f-9114ce9376a1'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call send_email_tool.', 'tool_calls': [{'id': 'fc_ed607c11-0e20-4c4b-887b-3ea37d6702ac', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 174, 'total_tokens': 220, 'completion_time': 0.046824785, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.011752479, 'prompt_tokens_details': None, 'queue_time': 0.308267007, 'total_time': 0.058577264}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_3cdd24b83b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id=

In [36]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Response: {result}")
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Response: {'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='c11a907b-9978-4033-a81f-9114ce9376a1'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call send_email_tool.', 'tool_calls': [{'id': 'fc_ed607c11-0e20-4c4b-887b-3ea37d6702ac', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 174, 'total_tokens': 220, 'completion_time': 0.046824785, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.011752479, 'prompt_tokens_details': None, 'queue_time': 0.308267007, 'total_time': 0.058577264}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_3cdd24b83b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': No

In [37]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='c11a907b-9978-4033-a81f-9114ce9376a1'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call send_email_tool.', 'tool_calls': [{'id': 'fc_ed607c11-0e20-4c4b-887b-3ea37d6702ac', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 174, 'total_tokens': 220, 'completion_time': 0.046824785, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.011752479, 'prompt_tokens_details': None, 'queue_time': 0.308267007, 'total_time': 0.058577264}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_3cdd24b83b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id=

### Reject

In [50]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model=model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [51]:
config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config)

In [52]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='a26baf28-85c2-4d78-be10-e01dcd0379ca'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call send_email_tool with body, recipient, subject.', 'tool_calls': [{'id': 'fc_eabbbb2d-def6-4f26-93fd-43e808cc581b', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 174, 'total_tokens': 226, 'completion_time': 0.054506757, 'completion_tokens_details': {'reasoning_tokens': 15}, 'prompt_time': 0.008488348, 'prompt_tokens_details': None, 'queue_time': 0.340309813, 'total_time': 0.062995105}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_a4315eb300', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None,

In [53]:
# Step 2: Reject
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: I’m sorry, but I wasn’t able to send that email.  
Would you like me to try again, or is there anything else I can help you with?


In [54]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='a26baf28-85c2-4d78-be10-e01dcd0379ca'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call send_email_tool with body, recipient, subject.', 'tool_calls': [{'id': 'fc_eabbbb2d-def6-4f26-93fd-43e808cc581b', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 174, 'total_tokens': 226, 'completion_time': 0.054506757, 'completion_tokens_details': {'reasoning_tokens': 15}, 'prompt_time': 0.008488348, 'prompt_tokens_details': None, 'queue_time': 0.340309813, 'total_time': 0.062995105}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_a4315eb300', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None,

### Editing

In [55]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model=model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [56]:
config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config
)

In [57]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='687e87a8-73bd-420a-892b-1edf41493c64'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "The user wants to send an email to wrong@email.com with subject 'Test' and body 'Hello'. We have a function send_email_tool. We need to call that.", 'tool_calls': [{'id': 'fc_577b1013-f501-4448-90af-35b22f545019', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@email.com","subject":"Test"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 70, 'prompt_tokens': 172, 'total_tokens': 242, 'completion_time': 0.073469434, 'completion_tokens_details': {'reasoning_tokens': 36}, 'prompt_time': 0.009966571, 'prompt_tokens_details': None, 'queue_time': 0.334041295, 'total_time': 0.083436005}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_3023a70d60', 'serv

In [58]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"✏️ Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...
✏️ Result: I’ve noted that the email was sent to **correct@email.com** with the subject **“Corrected Subject”** and body **“This was edited by human before sending.”** If you need to send a different message or have any other requests, just let me know!


In [59]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='687e87a8-73bd-420a-892b-1edf41493c64'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "The user wants to send an email to wrong@email.com with subject 'Test' and body 'Hello'. We have a function send_email_tool. We need to call that.", 'tool_calls': [{'id': 'fc_577b1013-f501-4448-90af-35b22f545019', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@email.com","subject":"Test"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 70, 'prompt_tokens': 172, 'total_tokens': 242, 'completion_time': 0.073469434, 'completion_tokens_details': {'reasoning_tokens': 36}, 'prompt_time': 0.009966571, 'prompt_tokens_details': None, 'queue_time': 0.334041295, 'total_time': 0.083436005}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_3023a70d60', 'serv